In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# =====================================
# LOAD DATA
# =====================================

loan_data = pd.read_csv("cleaned_loan_data.csv")

# =====================================
# CLEAN COLUMN NAMES
# =====================================

loan_data.columns = (
    loan_data.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

# =====================================
# DROP ID COLUMNS
# =====================================

loan_data.drop(
    ['loan_id', 'customer_id'],
    axis=1,
    inplace=True
)

# =====================================
# CLEAN years_in_current_job
# =====================================

loan_data['years_in_current_job'] = (
    loan_data['years_in_current_job']
    .str.replace('years', '', regex=False)
    .str.replace('year', '', regex=False)
    .str.replace('+', '', regex=False)
    .str.replace('< 1', '0', regex=False)
    .str.strip()
)

loan_data['years_in_current_job'] = pd.to_numeric(
    loan_data['years_in_current_job'],
    errors='coerce'
)

loan_data['years_in_current_job'].fillna(
    loan_data['years_in_current_job'].median(),
    inplace=True
)

# =====================================
# FEATURE ENGINEERING
# =====================================

loan_data['financial_burden'] = (
    loan_data['monthly_debt']
    /
    loan_data['annual_income']
)

loan_data['credit_utilization'] = (
    loan_data['current_credit_balance']
    /
    loan_data['maximum_open_credit']
)

loan_data['debt_per_account'] = (
    loan_data['monthly_debt']
    /
    loan_data['number_of_open_accounts']
)

# =====================================
# HANDLE INF VALUES
# =====================================

loan_data.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

loan_data.fillna(
    loan_data.median(numeric_only=True),
    inplace=True
)

# =====================================
# ENCODE TARGET
# =====================================

le = LabelEncoder()

loan_data['risk_encoded'] = le.fit_transform(
    loan_data['risk_category']
)

# =====================================
# FEATURES & TARGET
# =====================================

X = loan_data.drop(
    ['risk_category', 'risk_encoded'],
    axis=1
)

y = loan_data['risk_encoded']

# =====================================
# ONE HOT ENCODING
# =====================================

X = pd.get_dummies(
    X,
    drop_first=True
)

# =====================================
# CONVERT TO FLOAT
# =====================================

X = X.astype(float)

# =====================================
# SAVE FEATURE COLUMNS
# =====================================

model_columns = X.columns

pickle.dump(
    model_columns,
    open("model_columns.pkl", "wb")
)

# =====================================
# TRAIN TEST SPLIT
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =====================================
# TRAIN MODEL
# =====================================

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

# =====================================
# EVALUATE
# =====================================

pred = rf_model.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, pred)
)

# =====================================
# SAVE MODEL
# =====================================

pickle.dump(
    rf_model,
    open("risk_model.pkl", "wb")
)

print("MODEL TRAINED & SAVED SUCCESSFULLY")

C:\Users\muthamilharasi\AppData\Local\Temp\ipykernel_12528\2391011829.py:55: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  loan_data['years_in_current_job'].fillna(


Accuracy: 0.9960019990004998
MODEL TRAINED & SAVED SUCCESSFULLY
